# Figure 4 — Epicenter connectivity and clinical correlations

- **SubA**: Chord diagram of correlations between epicenter-seeded connectivity and clinical scales (ADOS, etc.).
- **SubB/C**: Scatter plots (with Pearson r, p-value, FDR correction) of clinical scores against epicenter-related connectivity.

## Panel SubA — Chord diagrams

In [ ]:
"""
Panel A: chord diagrams of epicenter-connectivity correlations with ADOS.

For each dataset (ABIDE-II, CABIC):
  1. Load subject info and per-subject MIND networks (Subtype L)
  2. Extract the 28 epicenter-to-epicenter connections and ComBat-harmonize across sites
  3. Partial Spearman correlation of each connection with every ADOS scale (controlling age, sex)
  4. FDR correction per scale
  5. Draw a connectivity-circle (chord) diagram for the ADOS total scale
"""
import os
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from statsmodels.sandbox.stats.multicomp import multipletests
from tqdm import tqdm
import matplotlib.pyplot as plt
from neuroCombat import neuroCombat
from mne_connectivity.viz import plot_connectivity_circle
import warnings
warnings.filterwarnings("ignore")

# --- Configuration ---
TARGET_SUBTYPE = 0
MIN_SAMPLE_SIZE = 20
GLOBAL_CONFIG = {'DPI': 300}
SUBTYPE_COL = 'SUBTYPE_LABEL'
ID_COL = 'SUBID'
SITE_COL = 'SITE'

TARGET_REGIONS = [
    'rh_caudalanteriorcingulate', 'lh_caudalanteriorcingulate',
    'rh_temporalpole', 'lh_temporalpole',
    'rh_insula', 'rh_frontalpole', 'lh_frontalpole',
    'rh_cuneus'
]
REGION_LABELS = {
    'rh_caudalanteriorcingulate': 'R_cACC', 'lh_caudalanteriorcingulate': 'L_cACC',
    'rh_temporalpole': 'R_TP', 'lh_temporalpole': 'L_TP',
    'rh_insula': 'R_Insula', 'lh_frontalpole': 'L_FP',
    'rh_frontalpole': 'R_FP', 'rh_cuneus': 'R_Cuneus'
}
# Fixed clockwise node order, symmetric about the vertical axis
FIXED_NODE_ORDER = ['L_cACC', 'R_cACC', 'R_Insula', 'R_Cuneus', 'L_FP', 'R_FP', 'L_TP', 'R_TP']

DATASET_CONFIGS = {
    'ABIDE2': {'path': 'data/ABIDE2_AGE_Sub.xlsx',
               'ados_cols': ['ADOS_2_SOCAFFECT', 'ADOS_2_RRB', 'ADOS_2_TOTAL', 'ADOS_2_SEVERITY_TOTAL'],
               'total_scale': 'ADOS_2_TOTAL'},
    'CABIC': {'path': 'data/CABIC_AGE_Sub.xlsx',
              'ados_cols': ['ADOS_SOCI', 'ADOS_COMM', 'ADOS_SA', 'ADOS_RRB', 'ADOS_TOTAL'],
              'total_scale': 'ADOS_TOTAL'},
}


def get_sig_stars(p_val):
    if p_val < 0.001:
        return '***'
    elif p_val < 0.01:
        return '**'
    elif p_val < 0.05:
        return '*'
    else:
        return ''


def partial_spearmanr(x, y, covars):
    """Partial Spearman correlation between x and y controlling for covars."""
    reg_x = LinearRegression().fit(covars, x)
    residual_x = x - reg_x.predict(covars)
    reg_y = LinearRegression().fit(covars, y)
    residual_y = y - reg_y.predict(covars)
    r, p = spearmanr(residual_x, residual_y)
    return r, p


def process_dataset(ds_name, cfg):
    """Load MIND networks -> ComBat -> partial Spearman vs ADOS -> FDR."""
    print(f"\n{'='*70}\n   Processing {ds_name}\n{'='*70}")
    df_info = pd.read_excel(cfg['path'])
    df_info[ID_COL] = df_info[ID_COL].astype(str).str.strip()
    df_sub = df_info[df_info[SUBTYPE_COL] == TARGET_SUBTYPE].copy()
    if ds_name == 'CABIC':
        if 'ADOS_SA' in df_sub.columns and 'ADOS_RRB' in df_sub.columns:
            df_sub['ADOS_TOTAL'] = (pd.to_numeric(df_sub['ADOS_SA'], errors='coerce')
                                    + pd.to_numeric(df_sub['ADOS_RRB'], errors='coerce'))
    sample_path = df_sub['aparc'].dropna().iloc[0]
    cols = pd.read_csv(sample_path, sep=None, engine='python').columns.tolist()
    region_idx = [cols.index(r) for r in TARGET_REGIONS if r in cols]
    c_names, c_pairs = [], []
    for i in range(len(TARGET_REGIONS)):
        for j in range(i + 1, len(TARGET_REGIONS)):
            c_names.append(f"{TARGET_REGIONS[i]}__{TARGET_REGIONS[j]}")
            c_pairs.append((region_idx[i], region_idx[j]))
    conn_list, meta_list = [], []
    for _, row in tqdm(df_sub.iterrows(), total=len(df_sub), desc=f"{ds_name} loading"):
        p = row['aparc']
        if pd.isna(p) or not os.path.exists(p):
            continue
        try:
            mat = pd.read_csv(p, sep=None, engine='python').values
            conn_list.append([mat[i, j] for i, j in c_pairs])
            meta_list.append(row)
        except:
            continue
    X_raw = np.array(conn_list)
    df_meta = pd.DataFrame(meta_list).reset_index(drop=True)
    site_counts = df_meta[SITE_COL].value_counts()
    valid_sites = site_counts[site_counts >= 2].index.tolist()
    mask = df_meta[SITE_COL].isin(valid_sites)
    df_meta, X_raw = df_meta[mask].reset_index(drop=True), X_raw[mask.values]
    covars = pd.DataFrame({
        'batch': df_meta[SITE_COL].values,
        'age': pd.to_numeric(df_meta['AGE'], errors='coerce').fillna(df_meta['AGE'].mean()).values,
        'sex': df_meta['SEX'].map({'M': 0, 'F': 1, 'MALE': 0, 'FEMALE': 1}).fillna(0).values,
    })
    X_corrected = neuroCombat(dat=X_raw.T, covars=covars, batch_col='batch',
                              continuous_cols=['age'], categorical_cols=['sex'])['data'].T
    df_final = pd.DataFrame(X_corrected, columns=c_names)
    df_final[ID_COL] = df_meta[ID_COL].values
    df_meta['SEX_code'] = df_meta['SEX'].map({'M': 0, 'F': 1, 'MALE': 0, 'FEMALE': 1}).fillna(0).astype(int)
    valid_ados = [c for c in cfg['ados_cols'] if c in df_meta.columns]
    df_merged = pd.merge(df_final, df_meta[[ID_COL, 'AGE', 'SEX_code'] + valid_ados], on=ID_COL).reset_index(drop=True)
    conn_results = []
    for scale in valid_ados:
        y_series = pd.to_numeric(df_merged[scale], errors='coerce')
        for conn in c_names:
            x_series = pd.to_numeric(df_merged[conn], errors='coerce')
            temp = pd.concat([x_series, y_series, df_merged['AGE'], df_merged['SEX_code']], axis=1).dropna()
            if len(temp) >= MIN_SAMPLE_SIZE and temp.iloc[:, 1].std() > 1e-6:
                r, p = partial_spearmanr(temp.iloc[:, 0].values, temp.iloc[:, 1].values, temp.iloc[:, 2:].values)
                conn_results.append({'Connection': conn, 'Clinical_Scale': scale, 'Partial_R': r, 'P_Value': p})
    df_conn_raw = pd.DataFrame(conn_results)
    corrected = []
    for scale in df_conn_raw['Clinical_Scale'].unique():
        sub = df_conn_raw[df_conn_raw['Clinical_Scale'] == scale].copy()
        _, p_fdr, _, _ = multipletests(sub['P_Value'], method='fdr_bh')
        sub['P_FDR'] = p_fdr
        sub['Sig'] = sub['P_FDR'].apply(get_sig_stars)
        corrected.append(sub)
    return pd.concat(corrected).sort_values('P_Value')


def plot_chord_png(df_results, clinical_scale, save_path, dataset_label):
    """Chord diagram of the partial correlations for one clinical scale."""
    print(f"\n--- Plotting Chord: {save_path} ---")
    df_scale = df_results[df_results['Clinical_Scale'] == clinical_scale].copy()
    n_nodes = len(FIXED_NODE_ORDER)
    node_to_idx = {name: i for i, name in enumerate(FIXED_NODE_ORDER)}
    con_matrix = np.zeros((n_nodes, n_nodes))
    for _, row in df_scale.iterrows():
        parts = row['Connection'].split('__')
        if len(parts) == 2:
            source = REGION_LABELS.get(parts[0], parts[0])
            target = REGION_LABELS.get(parts[1], parts[1])
            if source in node_to_idx and target in node_to_idx:
                i1, i2 = node_to_idx[source], node_to_idx[target]
                con_matrix[i1, i2] = row['Partial_R']
                con_matrix[i2, i1] = row['Partial_R']
    fig, ax = plt.subplots(figsize=(7, 7), facecolor='white', subplot_kw=dict(polar=True))
    plot_connectivity_circle(con=con_matrix, node_names=FIXED_NODE_ORDER,
                             n_lines=None, vmin=-0.4, vmax=0.4,
                             colormap='RdBu_r', facecolor='white', textcolor='black',
                             node_colors=['#E15759'] * n_nodes, node_edgecolor='white',
                             linewidth=2.5, ax=ax, show=False, fontsize_names=10)
    ax.set_position([0.05, 0.05, 0.76, 0.90])
    if len(fig.axes) > 1:
        fig.delaxes(fig.axes[1])          # remove the default colorbar
    cbar_ax = fig.add_axes([0.79, 0.29, 0.022, 0.42])
    sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.4, vmax=0.4))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')
    cbar.ax.tick_params(labelsize=9)
    cbar.outline.set_visible(False)
    # Make node labels horizontal and aligned
    for text_obj in ax.texts:
        pos_theta = (text_obj.get_position()[0] + np.pi) % (2 * np.pi) - np.pi
        text_obj.set_rotation(0)
        text_obj.set_fontsize(13)
        tol = 5.0 * np.pi / 180.0
        if abs(pos_theta - np.pi / 2) < tol or abs(pos_theta + np.pi / 2) < tol:
            text_obj.set_horizontalalignment('center')
            text_obj.set_verticalalignment('bottom' if pos_theta > 0 else 'top')
        elif -np.pi / 2 < pos_theta < np.pi / 2:
            text_obj.set_horizontalalignment('left')
            text_obj.set_verticalalignment('center')
        else:
            text_obj.set_horizontalalignment('right')
            text_obj.set_verticalalignment('center')
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=GLOBAL_CONFIG['DPI'], bbox_inches='tight')
    plt.show()
    print(f"  Saved: {save_path}")


# --- Main loop: ABIDE-II and CABIC ---
for ds_name, cfg in DATASET_CONFIGS.items():
    df_results = process_dataset(ds_name, cfg)
    total_scale = cfg['total_scale']
    df_total = df_results[df_results['Clinical_Scale'] == total_scale]
    n_total = len(df_total)
    n_neg = (df_total['Partial_R'] < 0).sum()
    n_pos = (df_total['Partial_R'] > 0).sum()
    n_sig = (df_total['P_FDR'] < 0.05).sum()
    print(f"\n  [{ds_name}] {total_scale} summary: total={n_total}, neg={n_neg} "
          f"({n_neg/n_total*100:.1f}%), pos={n_pos} ({n_pos/n_total*100:.1f}%), FDR-sig={n_sig}")
    if total_scale in df_results['Clinical_Scale'].unique():
        plot_chord_png(df_results, total_scale, f'Fig4/SubA_{ds_name}.png', ds_name)
    else:
        print(f"  ! {ds_name}: scale {total_scale} not found")

## Panel SubB — Target connection scatter plots

In [ ]:
"""
Panel B: partial Spearman correlation of all 28 epicenter connections with
ADOS Total (controlling age, sex). After FDR correction, plot only the
dataset-specific target connection (rh_cACC__rh_Insula for ABIDE-II,
rh_Insula__rh_Cuneus for CABIC). Self-contained cell.
"""
import os
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from statsmodels.sandbox.stats.multicomp import multipletests
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from neuroCombat import neuroCombat
import warnings
warnings.filterwarnings("ignore")

# --- Configuration ---
TARGET_SUBTYPE = 0
MIN_SAMPLE_SIZE = 20
SUBTYPE_MAPPING = {0: 'L', 1: 'H'}
CURRENT_SUBTYPE_LABEL = SUBTYPE_MAPPING.get(TARGET_SUBTYPE, str(TARGET_SUBTYPE))
GLOBAL_CONFIG = {'DPI': 300, 'SCATTER_W': 9, 'SCATTER_H': 7,
                 'SCATTER_MAIN': 26, 'SCATTER_LABEL': 24, 'SCATTER_TICK': 22}
SUBTYPE_COL = 'SUBTYPE_LABEL'
ID_COL = 'SUBID'
SITE_COL = 'SITE'

TARGET_REGIONS = [
    'rh_caudalanteriorcingulate', 'lh_caudalanteriorcingulate',
    'rh_temporalpole', 'lh_temporalpole',
    'rh_insula', 'rh_cuneus', 'rh_frontalpole', 'lh_frontalpole'
]

# ABIDE-II and CABIC use different target connections
TARGET_CONNS = {
    'ABIDE-II': 'rh_caudalanteriorcingulate__rh_insula',
    'CABIC': 'rh_insula__rh_cuneus',
}


def get_sig_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''


def partial_spearmanr(x, y, covars):
    """Partial Spearman correlation controlling for covars."""
    reg_x = LinearRegression().fit(covars, x)
    residual_x = x - reg_x.predict(covars)
    reg_y = LinearRegression().fit(covars, y)
    residual_y = y - reg_y.predict(covars)
    r, p = spearmanr(residual_x, residual_y)
    return r, p


def load_and_combat_all(dataset, subject_subtype_path, ados_cols, target_subtype=0):
    """Load per-subject connections, ComBat-harmonize, return connections + AGE + SEX_code + ADOS."""
    print(f"\n  Processing {dataset} (Subtype {SUBTYPE_MAPPING.get(target_subtype, str(target_subtype))})")
    df_info = pd.read_excel(subject_subtype_path)
    df_info[ID_COL] = df_info[ID_COL].astype(str).str.strip()
    df_sub = df_info[df_info[SUBTYPE_COL] == target_subtype].copy()
    if dataset == 'CABIC':
        if 'ADOS_SA' in df_sub.columns and 'ADOS_RRB' in df_sub.columns:
            df_sub['ADOS_TOTAL'] = (pd.to_numeric(df_sub['ADOS_SA'], errors='coerce')
                                    + pd.to_numeric(df_sub['ADOS_RRB'], errors='coerce'))
    sample_path = df_sub['aparc'].dropna().iloc[0]
    cols = pd.read_csv(sample_path, sep=None, engine='python').columns.tolist()
    region_idx = [cols.index(r) for r in TARGET_REGIONS if r in cols]
    c_names, c_pairs = [], []
    for i in range(len(TARGET_REGIONS)):
        for j in range(i + 1, len(TARGET_REGIONS)):
            c_names.append(f"{TARGET_REGIONS[i]}__{TARGET_REGIONS[j]}")
            c_pairs.append((region_idx[i], region_idx[j]))
    conn_list, meta_list = [], []
    for _, row in tqdm(df_sub.iterrows(), total=len(df_sub), desc=f"{dataset} loading"):
        p = row['aparc']
        if pd.isna(p) or not os.path.exists(p):
            continue
        try:
            mat = pd.read_csv(p, sep=None, engine='python').values
            conn_list.append([mat[i, j] for i, j in c_pairs])
            meta_list.append(row)
        except:
            continue
    X_raw = np.array(conn_list)
    df_meta = pd.DataFrame(meta_list).reset_index(drop=True)
    site_counts = df_meta[SITE_COL].value_counts()
    valid_sites = site_counts[site_counts >= 2].index.tolist()
    mask = df_meta[SITE_COL].isin(valid_sites)
    df_meta, X_raw = df_meta[mask].reset_index(drop=True), X_raw[mask.values]
    covars = pd.DataFrame({
        'batch': df_meta[SITE_COL].values,
        'age': pd.to_numeric(df_meta['AGE'], errors='coerce').fillna(df_meta['AGE'].mean()).values,
        'sex': df_meta['SEX'].map({'M': 0, 'F': 1, 'MALE': 0, 'FEMALE': 1}).fillna(0).values,
    })
    X_corrected = neuroCombat(dat=X_raw.T, covars=covars, batch_col='batch',
                              continuous_cols=['age'], categorical_cols=['sex'])['data'].T
    df_final = pd.DataFrame(X_corrected, columns=c_names)
    df_final[ID_COL] = df_meta[ID_COL].values
    valid_ados = [c for c in ados_cols if c in df_meta.columns]
    df_meta['SEX_code'] = df_meta['SEX'].map({'M': 0, 'F': 1, 'MALE': 0, 'FEMALE': 1}).fillna(0).astype(int)
    df_merged = pd.merge(df_final, df_meta[[ID_COL, 'AGE', 'SEX_code'] + valid_ados], on=ID_COL).reset_index(drop=True)
    return df_merged, c_names


def plot_scatter_target(data, x_col, y_col, r_val, p_val, p_fdr, save_path, x_label, y_label, dataset_label):
    """Scatter plot annotated with the partial Spearman stats (stars from FDR)."""
    plt.figure(figsize=(GLOBAL_CONFIG['SCATTER_W'], GLOBAL_CONFIG['SCATTER_H']))
    sns.set_style("whitegrid")
    sns.regplot(x=x_col, y=y_col, data=data,
                scatter_kws={'alpha': 0.7, 's': 90, 'color': '#4C72B0', 'edgecolor': 'white'},
                line_kws={'color': '#C44E52', 'lw': 3}, ci=95)
    n = len(data)
    sig_str = get_sig_stars(p_fdr)
    plt.title(f'{dataset_label}  |  N={n}', fontsize=GLOBAL_CONFIG['SCATTER_MAIN'], pad=15)
    plt.xlabel(x_label, fontsize=GLOBAL_CONFIG['SCATTER_LABEL'])
    plt.ylabel(y_label, fontsize=GLOBAL_CONFIG['SCATTER_LABEL'])
    plt.xticks(fontsize=GLOBAL_CONFIG['SCATTER_TICK'])
    plt.yticks(fontsize=GLOBAL_CONFIG['SCATTER_TICK'])
    stat_text = (f"Partial $r_s$ = {r_val:.3f}{sig_str}\n"
                 f"$P$ = {p_val:.4f}\n$P_{{FDR}}$ = {p_fdr:.4f}")
    plt.text(0.05, 0.95, stat_text, transform=plt.gca().transAxes, verticalalignment='top',
             bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="#AAAAAA", alpha=0.9),
             fontsize=GLOBAL_CONFIG['SCATTER_TICK'] - 2)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=GLOBAL_CONFIG['DPI'], bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")


# --- Partial Spearman: 28 connections x ADOS Total, FDR corrected ---
print("=" * 70)
print("  Partial Spearman: 28 epicenter connections x ADOS Total (controlling age, sex)")
print("=" * 70)

DATASETS = [
    ('ABIDE-II', 'data/ABIDE2_AGE_Sub.xlsx', 'ADOS_2_TOTAL'),
    ('CABIC', 'data/CABIC_AGE_Sub.xlsx', 'ADOS_TOTAL'),
]

for ds_name, ds_path, ds_scale in DATASETS:
    print(f"\n--- {ds_name} ---")
    df_data, conn_list = load_and_combat_all(ds_name, ds_path, [ds_scale])
    y_series = pd.to_numeric(df_data[ds_scale], errors='coerce')
    all_res = []
    for conn in conn_list:
        x_series = pd.to_numeric(df_data[conn], errors='coerce')
        temp = pd.concat([x_series, y_series, df_data['AGE'], df_data['SEX_code']], axis=1).dropna()
        if len(temp) >= MIN_SAMPLE_SIZE and temp.iloc[:, 1].std() > 1e-6:
            r, p = partial_spearmanr(temp.iloc[:, 0].values, temp.iloc[:, 1].values, temp.iloc[:, 2:].values)
            all_res.append({'Connection': conn, 'Partial_Rs': r, 'P_Value': p})
    df_res = pd.DataFrame(all_res)
    _, p_fdr, _, _ = multipletests(df_res['P_Value'], method='fdr_bh')
    df_res['P_FDR'] = p_fdr
    df_res['Sig'] = df_res['P_FDR'].apply(get_sig_stars)
    df_res = df_res.sort_values('P_Value')
    print(f"\n  {'Connection':45s} {'Rs':>7s} {'P_raw':>8s} {'P_FDR':>8s}")
    print("  " + "-" * 72)
    for _, row in df_res.iterrows():
        print(f"  {row['Connection']:45s} {row['Partial_Rs']:+7.4f} {row['P_Value']:8.4f} {row['P_FDR']:8.4f} {row['Sig']}")

    # Extract the dataset-specific target connection
    TARGET_CONN = TARGET_CONNS[ds_name]
    target_row = df_res[df_res['Connection'] == TARGET_CONN]
    if len(target_row) > 0:
        r_t, p_t, fdr_t = (target_row.iloc[0]['Partial_Rs'], target_row.iloc[0]['P_Value'],
                           target_row.iloc[0]['P_FDR'])
        n_t = len(df_data[[TARGET_CONN, ds_scale, 'AGE', 'SEX_code']].apply(pd.to_numeric, errors='coerce').dropna())
        print(f"\n  * {TARGET_CONN}: rs={r_t:+.4f}, p={p_t:.4f}, FDR={fdr_t:.4f} {target_row.iloc[0]['Sig']}")
        temp = df_data[[TARGET_CONN, ds_scale, 'AGE', 'SEX_code']].apply(pd.to_numeric, errors='coerce').dropna()
        suffix = 'ABIDE2' if ds_name == 'ABIDE-II' else 'CABIC'
        x_label = 'rh_Insula - rh_Cuneus Connectivity' if ds_name == 'CABIC' else 'rh_cACC - rh_Insula Connectivity'
        plot_scatter_target(temp, TARGET_CONN, ds_scale, r_t, p_t, fdr_t,
                            f'Fig4/SubB_{suffix}.png', x_label, 'ADOS Total', ds_name)

print(f"\n{'='*70}")
print("  Done: Fig4/SubB_ABIDE2.png, Fig4/SubB_CABIC.png")
print(f"{'='*70}")

## Panel SubC — Mean epicenter score scatter plots

In [ ]:
"""
Panel C: partial Spearman correlation of the mean epicenter-connectivity score
with three ADOS scales (controlling age, sex). After FDR correction, plot only
the ADOS RRB scatter for each dataset. Self-contained cell.
"""
import os
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from statsmodels.sandbox.stats.multicomp import multipletests
from neuroCombat import neuroCombat
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# --- Configuration ---
MIN_SAMPLE_SIZE = 20
TARGET_SUBTYPE = 0
SUBTYPE_COL = 'SUBTYPE_LABEL'
ID_COL = 'SUBID'
SITE_COL = 'SITE'
MEAN_SCORE_COL = 'Mean_Connectivity_Score'
GLOBAL_CONFIG = {'DPI': 300, 'SCATTER_W': 9, 'SCATTER_H': 7,
                 'SCATTER_MAIN': 26, 'SCATTER_LABEL': 24, 'SCATTER_TICK': 22}

TARGET_REGIONS = [
    'rh_caudalanteriorcingulate', 'lh_caudalanteriorcingulate',
    'rh_temporalpole', 'lh_temporalpole',
    'rh_insula', 'rh_frontalpole', 'lh_frontalpole',
    'rh_cuneus'
]

DATASETS = [
    ('ABIDE-II', 'data/ABIDE2_AGE_Sub.xlsx',
     [('ADOS_2_TOTAL', 'ADOS Total'), ('ADOS_2_SOCAFFECT', 'ADOS Social Affect'), ('ADOS_2_RRB', 'ADOS RRB')],
     'ADOS_2_RRB'),
    ('CABIC', 'data/CABIC_AGE_Sub.xlsx',
     [('ADOS_TOTAL', 'ADOS Total'), ('ADOS_SA', 'ADOS Social Affect'), ('ADOS_RRB', 'ADOS RRB')],
     'ADOS_RRB'),
]


def get_sig_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''


def partial_spearmanr(x, y, covars):
    """Partial Spearman correlation controlling for covars."""
    reg_x = LinearRegression().fit(covars, x)
    residual_x = x - reg_x.predict(covars)
    reg_y = LinearRegression().fit(covars, y)
    residual_y = y - reg_y.predict(covars)
    r, p = spearmanr(residual_x, residual_y)
    return r, p


def load_mean_score_data(dataset_name, ados_cols):
    """Load per-subject connections, ComBat-harmonize, return mean epicenter score + ADOS."""
    path = 'data/ABIDE2_AGE_Sub.xlsx' if dataset_name == 'ABIDE-II' else 'data/CABIC_AGE_Sub.xlsx'
    df_info = pd.read_excel(path)
    df_info[ID_COL] = df_info[ID_COL].astype(str).str.strip()
    df_sub = df_info[df_info[SUBTYPE_COL] == TARGET_SUBTYPE].copy()
    if dataset_name == 'CABIC':
        if 'ADOS_SA' in df_sub.columns and 'ADOS_RRB' in df_sub.columns:
            df_sub['ADOS_TOTAL'] = (pd.to_numeric(df_sub['ADOS_SA'], errors='coerce')
                                    + pd.to_numeric(df_sub['ADOS_RRB'], errors='coerce'))
    sample_path = df_sub['aparc'].dropna().iloc[0]
    cols = pd.read_csv(sample_path, sep=None, engine='python').columns.tolist()
    region_idx = [cols.index(r) for r in TARGET_REGIONS if r in cols]
    c_names, c_pairs = [], []
    for i in range(len(TARGET_REGIONS)):
        for j in range(i + 1, len(TARGET_REGIONS)):
            c_names.append(f"{TARGET_REGIONS[i]}__{TARGET_REGIONS[j]}")
            c_pairs.append((region_idx[i], region_idx[j]))
    conn_list, meta_list = [], []
    for _, row in tqdm(df_sub.iterrows(), total=len(df_sub), desc=f"{dataset_name} loading"):
        p = row['aparc']
        if pd.isna(p) or not os.path.exists(p):
            continue
        try:
            mat = pd.read_csv(p, sep=None, engine='python').values
            conn_list.append([mat[i, j] for i, j in c_pairs])
            meta_list.append(row)
        except:
            continue
    X_raw = np.array(conn_list)
    df_meta = pd.DataFrame(meta_list).reset_index(drop=True)
    site_counts = df_meta[SITE_COL].value_counts()
    valid_sites = site_counts[site_counts >= 2].index.tolist()
    mask = df_meta[SITE_COL].isin(valid_sites)
    df_meta, X_raw = df_meta[mask].reset_index(drop=True), X_raw[mask.values]
    covars = pd.DataFrame({
        'batch': df_meta[SITE_COL].values,
        'age': pd.to_numeric(df_meta['AGE'], errors='coerce').fillna(df_meta['AGE'].mean()).values,
        'sex': df_meta['SEX'].map({'M': 0, 'F': 1, 'MALE': 0, 'FEMALE': 1}).fillna(0).values,
    })
    X_corrected = neuroCombat(dat=X_raw.T, covars=covars, batch_col='batch',
                              continuous_cols=['age'], categorical_cols=['sex'])['data'].T
    df_final = pd.DataFrame(X_corrected, columns=c_names)
    df_final[MEAN_SCORE_COL] = np.mean(X_corrected, axis=1)
    df_final[ID_COL] = df_meta[ID_COL].values
    valid_scales = [c for c in ados_cols if c in df_meta.columns]
    df_meta['SEX_code'] = df_meta['SEX'].map({'M': 0, 'F': 1, 'MALE': 0, 'FEMALE': 1}).fillna(0).astype(int)
    df_merged = pd.merge(df_final[[MEAN_SCORE_COL, ID_COL]],
                         df_meta[[ID_COL, 'AGE', 'SEX_code'] + valid_scales], on=ID_COL).reset_index(drop=True)
    return df_merged


def plot_mean_scatter(data, x_col, y_col, r_val, p_val, p_fdr, save_path, y_label, dataset_label):
    """Scatter plot annotated with stats; stars shown only when FDR < 0.05."""
    plt.figure(figsize=(GLOBAL_CONFIG['SCATTER_W'], GLOBAL_CONFIG['SCATTER_H']))
    sns.set_style("whitegrid")
    sns.regplot(x=x_col, y=y_col, data=data,
                scatter_kws={'alpha': 0.7, 's': 90, 'color': '#4C72B0', 'edgecolor': 'white'},
                line_kws={'color': '#C44E52', 'lw': 3}, ci=95)
    n = len(data)
    sig_str = get_sig_stars(p_fdr)
    plt.title(f'{dataset_label}  |  N={n}', fontsize=GLOBAL_CONFIG['SCATTER_MAIN'], pad=15)
    plt.xlabel('Mean Epicenter Connectivity Score', fontsize=GLOBAL_CONFIG['SCATTER_LABEL'])
    plt.ylabel(y_label, fontsize=GLOBAL_CONFIG['SCATTER_LABEL'])
    plt.xticks(fontsize=GLOBAL_CONFIG['SCATTER_TICK'])
    plt.yticks(fontsize=GLOBAL_CONFIG['SCATTER_TICK'])
    stat_text = (f"Partial $r_s$ = {r_val:.3f}{sig_str}\n"
                 f"$P$ = {p_val:.4f}\n$P_{{FDR}}$ = {p_fdr:.4f}")
    plt.text(0.05, 0.95, stat_text, transform=plt.gca().transAxes, verticalalignment='top',
             bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="#AAAAAA", alpha=0.9),
             fontsize=GLOBAL_CONFIG['SCATTER_TICK'] - 2)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=GLOBAL_CONFIG['DPI'], bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved: {save_path}")


# --- Analysis: Mean_Score x 3 ADOS scales, FDR, plot RRB only ---
print("=" * 80)
print("  Mean_Score x 3 ADOS scales: partial Spearman (controlling age, sex), FDR corrected")
print("=" * 80)

for ds_name, ds_path, scales_with_labels, rrb_col in DATASETS:
    print(f"\n--- {ds_name} ---")
    scale_cols = [s for s, _ in scales_with_labels]
    df_data = load_mean_score_data(ds_name, scale_cols)
    results = []
    for scale_col, scale_label in scales_with_labels:
        if scale_col not in df_data.columns:
            continue
        temp = df_data[[MEAN_SCORE_COL, scale_col, 'AGE', 'SEX_code']].apply(pd.to_numeric, errors='coerce').dropna()
        if len(temp) >= MIN_SAMPLE_SIZE and temp[scale_col].std() > 1e-6:
            r, p = partial_spearmanr(temp[MEAN_SCORE_COL].values.reshape(-1, 1),
                                     temp[scale_col].values, temp[['AGE', 'SEX_code']].values)
            results.append({'Scale': scale_col, 'Label': scale_label, 'Partial_Rs': r, 'P_Value': p, 'N': len(temp)})
    df_res = pd.DataFrame(results)
    _, p_fdr, _, _ = multipletests(df_res['P_Value'], method='fdr_bh')
    df_res['P_FDR'] = p_fdr
    df_res['Sig'] = df_res['P_FDR'].apply(get_sig_stars)
    df_res = df_res.sort_values('P_Value')
    print(df_res[['Scale', 'Label', 'N', 'Partial_Rs', 'P_Value', 'P_FDR', 'Sig']].to_string(index=False))

    # Plot only the RRB scale
    rrb_row = df_res[df_res['Scale'] == rrb_col]
    if len(rrb_row) > 0:
        r0 = rrb_row.iloc[0]
        suffix = 'ABIDE2' if ds_name == 'ABIDE-II' else 'CABIC'
        plot_mean_scatter(df_data, MEAN_SCORE_COL, rrb_col, r0['Partial_Rs'], r0['P_Value'], r0['P_FDR'],
                          f'Fig4/SubC_{suffix}.png', 'ADOS RRB', ds_name)

print(f"\n{'='*80}")
print("  Done: Fig4/SubC_ABIDE2.png, Fig4/SubC_CABIC.png")
print(f"{'='*80}")